In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os
from scipy.signal import welch
from specula.mmlib.utils import get_pupil_mask

from utils import (calib_dir, result_dir, rec_noise_var, noise_sensitivities,
                   propagated_noise_var, get_noise_propagation, get_aliasing_power, get_atmo_coeffs, plot_rec_cov, get_rec)

In [ ]:
import matplotlib as mpl
# from cycler import cycler

cmap = mpl.colormaps['hot'] # 1. Choose your colormap (e.g., 'hot', 'jet', 'turbo')
colors = cmap(np.linspace(0.0, 1.0, 7)) # 2. Extract a set of colors from it (e.g., 6 distinct colors across the map)
# plt.rcParams['axes.prop_cycle'] = cycler(color=colors) # 3. Apply it to the default line cycle
plt.rcParams['image.cmap'] = 'magma' # 4. Change the default colormap for images/scatter plots
plt.rcParams.update({
    "text.usetex": True, # Use LaTeX to write all text
    "font.family": "serif", # Use a serif font
    "font.serif": ["Computer Modern Roman"] # Specifically use the classic LaTeX font
})
plt.rcParams['image.origin'] = 'lower'

In [ ]:
# calib_dir, result_dir: imported from utils

In [ ]:
phi = fits.getdata(os.path.join(result_dir,'res_ef.fits'))[:,1,:,:]
amp = np.sum(fits.getdata(os.path.join(result_dir,'res_ef.fits'))[:,0,:,:],axis=0) > 0

phi -= np.mean(phi[:,amp],axis=1)[:,None,None]

plt.figure()
plt.plot(np.std(phi[:,amp],axis=1))
plt.grid()

plt.figure()
plt.imshow(np.ma.masked_array(phi[10]-phi[11],mask=(1-amp).astype(bool)),vmin=-500,vmax=500)
plt.colorbar()

In [ ]:
frames_vis = fits.getdata(os.path.join(result_dir,'vis_frames.fits'))
frames_nir = fits.getdata(os.path.join(result_dir,'nir_frames.fits'))

plt.figure()
plt.imshow(np.mean(frames_vis,axis=0))

In [ ]:
tn = '20261104_105059' # mag = 7
tn = '20261104_105155' # mag = 6
tn = '20261104_105300','20261104_110824','20261104_111252' # mag = 10
tn = '20261104_111754' # mag = 10, thr = -5
tn = '20261104_112815' # mag = 10, no clipping
tn = '20261104_121113' # mag = 11, no clipping
tn = '20261104_130003' # mag = 11, no clipping
noisev = fits.getdata(os.path.join(result_dir,tn,'meas_vis.fits'))
noiser = fits.getdata(os.path.join(result_dir,tn,'meas_nir.fits'))
frames_vis = fits.getdata(os.path.join(result_dir,tn,'vis_frames.fits'))
frames_nir = fits.getdata(os.path.join(result_dir,tn,'nir_frames.fits'))

noise_vis = get_noise_propagation(rMod=2, nSubaps=10, nModes=54, wvl=900, sigma_e=0.5, flux=np.sum(np.mean(frames_vis,axis=0)))
noise_nir = get_noise_propagation(rMod=0, nSubaps=24, nModes=300, wvl=1300, sigma_e=0.5, flux=np.sum(np.mean(frames_nir,axis=0)))

xv = np.arange(len(noisev.T))+1
xr = np.arange(len(noiser.T))+1

plt.loglog(xv, np.std(noisev, axis=0))
plt.loglog(xv, noise_vis[:len(xv)],'--')
plt.loglog(xr, np.std(noiser, axis=0))
plt.loglog(xr, noise_nir[:len(xr)],'--')
plt.grid(which='both',alpha=0.4)

In [ ]:
nModes = 54
nSubaps = 10
rMod = 0
alias_var = get_aliasing_power(rMod, nSubaps, nModes, wvl=900)        # snapshot; fs=... for one frame at fs

plt.figure()
plt.loglog(alias_var)

In [ ]:
# noisev = fits.getdata(os.path.join(result_dir,'20261102_153912','noise_vis.fits'))
# noiser = fits.getdata(os.path.join(result_dir,'20261102_153912','noise_nir.fits'))
# sigv = fits.getdata(os.path.join(result_dir,'20261102_153912','sig_vis.fits'))
# sigr = fits.getdata(os.path.join(result_dir,'20261102_153912','sig_nir.fits'))
# x = np.arange(1300)+1

# plt.figure(figsize=(12,5))
# plt.subplot(1,2,1)
# plt.loglog(x,np.std(noisev,axis=0))
# plt.loglog(x,np.std(noiser,axis=0))
# plt.grid(which='both',alpha=0.4)
# plt.subplot(1,2,2)
# plt.loglog(x,np.std(sigr,axis=0))
# plt.loglog(x,np.std(sigv,axis=0))
# plt.grid(which='both',alpha=0.4)

In [ ]:
# f,psdv = welch(measv,fs=500,axis=0)
# f,psdr = welch(measr,fs=2000,axis=0)
# print(f.shape,psdv.shape)

# idxes = [1,10,100,1000]
# plt.figure()
# for idx in idxes:
#     plt.loglog(f[1:],psdv[1:,idx])
# for j,idx in enumerate(idxes):
#     plt.loglog(f[1:],psdr[1:,idx],'--',c=f'C{j}')
# plt.grid(which='both',alpha=0.4)

In [ ]:
rec0 = get_rec(0.0, 48, 1300, 1300)#*(900/1300)
rec05 = get_rec(0.5, 48, 1300, 1300)
rec1 = get_rec(1.0, 48, 1300, 1300)
rec2 = get_rec(2.0, 48, 1300, 1300)

plt.figure()
plt.loglog(np.arange(rec0.shape[0])+1,np.diag(rec0 @ rec0.T))
plt.loglog(np.arange(rec05.shape[0])+1,np.diag(rec05 @ rec05.T))
plt.loglog(np.arange(rec1.shape[0])+1,np.diag(rec1 @ rec1.T))
plt.loglog(np.arange(rec2.shape[0])+1,np.diag(rec2 @ rec2.T))
plt.grid(which='both',alpha=0.4)

In [ ]:
# wvl = 1300
# rec48x48 = get_rec(0.0, 48, 1300, wvl)
# rec36x36 = get_rec(0.0, 36, 800, wvl)
# rec24x24 = get_rec(0.0, 24, 300, wvl)
# rec14x14 = get_rec(0.0, 14, 100, wvl)
# rec10x10 = get_rec(0.0, 10, 54, wvl)

# plot_rec_cov: imported from utils
# plt.figure()
# plot_rec_cov(rec48x48,nSubap=48)
# plot_rec_cov(rec36x36,nSubap=36)
# plot_rec_cov(rec24x24,nSubap=24)
# plot_rec_cov(rec14x14,nSubap=14)
# plot_rec_cov(rec10x10,nSubap=10)
# plt.grid(which='both',alpha=0.4)